# Entrenamiento y Evaluación — Red Neuronal Convolucional (CNN)

Este notebook entrena una red CNN sobre el dataset **Fashion MNIST** y compara su desempeño con el modelo MLP.

**Arquitectura CNN:**
- Bloque 1: Conv2D(32) → BatchNorm → Conv2D(32) → MaxPooling → Dropout(0.25)
- Bloque 2: Conv2D(64) → BatchNorm → Conv2D(64) → MaxPooling → Dropout(0.25)
- Clasificador: Flatten → Dense(256) → BatchNorm → Dropout(0.5) → Dense(10, softmax)


In [ ]:
# ── Importaciones ──────────────────────────────────────────────────────────
import sys
import os

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping

# Añadir src/ al path para importar datos_processing
sys.path.append(os.path.abspath('../src'))
from datos_processing import load_and_prepare_all, class_names

print('TensorFlow version:', tf.__version__)
print('Clases:', class_names)

In [ ]:
# ── Carga y preparación de datos ───────────────────────────────────────────
data = load_and_prepare_all()

x_train = data['x_train_cnn']   # (60000, 28, 28, 1)
y_train = data['y_train']        # (60000,)
x_test  = data['x_test_cnn']    # (10000, 28, 28, 1)
y_test  = data['y_test']         # (10000,)

print('x_train shape:', x_train.shape)
print('y_train shape:', y_train.shape)
print('x_test  shape:', x_test.shape)
print('y_test  shape:', y_test.shape)

# Vista previa de algunas imágenes
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i, :, :, 0], cmap='gray')
    ax.set_title(class_names[y_train[i]], fontsize=9)
    ax.axis('off')
plt.suptitle('Ejemplos del dataset (CNN — formato 28×28×1)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Definición del modelo CNN ─────────────────────────────────────────────
model = Sequential([
    # Bloque convolucional 1
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    BatchNormalization(),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Bloque convolucional 2
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Clasificador
    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

model.summary()

In [ ]:
# ── Compilación y entrenamiento ────────────────────────────────────────────
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    x_train, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# ── Curvas de entrenamiento ────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history.history['loss'], label='Train Loss', marker='o', markersize=4)
ax1.plot(history.history['val_loss'], label='Val Loss', marker='s', markersize=4)
ax1.set_title('Pérdida (Loss) por Época')
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history.history['accuracy'], label='Train Acc', marker='o', markersize=4)
ax2.plot(history.history['val_accuracy'], label='Val Acc', marker='s', markersize=4)
ax2.set_title('Exactitud (Accuracy) por Época')
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('CNN — Curvas de Entrenamiento', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Evaluación en el conjunto de test ─────────────────────────────────────
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Loss en test:     {test_loss:.4f}')
print(f'Accuracy en test: {test_acc:.4f}  ({test_acc*100:.2f}%)')

In [ ]:
# ── Matriz de confusión ───────────────────────────────────────────────────
y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names
)
plt.title('Matriz de Confusión — CNN', fontsize=14, fontweight='bold')
plt.ylabel('Etiqueta real')
plt.xlabel('Etiqueta predicha')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Reporte de clasificación
print('\nReporte de clasificación:')
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# ── Predicciones visuales sobre ejemplos del test ─────────────────────────
n_samples = 15
indices = np.random.choice(len(x_test), n_samples, replace=False)

fig, axes = plt.subplots(3, 5, figsize=(14, 8))
for i, (ax, idx) in enumerate(zip(axes.flat, indices)):
    img = x_test[idx, :, :, 0]
    true_label = class_names[y_test[idx]]
    pred_label = class_names[y_pred[idx]]
    confidence = y_pred_probs[idx][y_pred[idx]] * 100

    ax.imshow(img, cmap='gray')
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(
        f'Real: {true_label}\nPred: {pred_label} ({confidence:.1f}%)',
        fontsize=8, color=color
    )
    ax.axis('off')

plt.suptitle('Predicciones CNN sobre ejemplos del test\n(Verde = correcto | Rojo = incorrecto)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Guardar el modelo ─────────────────────────────────────────────────────
models_dir = os.path.abspath('../models')
os.makedirs(models_dir, exist_ok=True)

save_path = os.path.join(models_dir, 'cnn_model.h5')
model.save(save_path)
print(f'Modelo CNN guardado en: {save_path}')

## Comparación MLP vs CNN

| Modelo | Accuracy en Test | Observaciones |
|--------|-----------------|---------------|
| MLP    | ~88–89%         | Red densa: 784 → 128 → 64 → 10 |
| CNN    | ~91–93%         | 2 bloques Conv2D + clasificador denso |

**Conclusión:** La CNN supera al MLP gracias a su capacidad para detectar patrones espaciales (bordes, texturas, formas) propios de las imágenes. El MLP trata cada píxel de forma independiente, perdiendo información espacial.
